# 09. Multimodal Fusion -> EEG + Eye Tracking

In [3]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# 1. Configuração de Caminhos dos Arquivos 
CAMINHO_EEG = '/workspaces/multimodal_fusion_eeg-et/reports/tabela_features_eeg_completa.csv'
CAMINHO_ET  = '/workspaces/multimodal_fusion_eeg-et/data/processed/dataset_features_et.csv'

ARQUIVO_SAIDA = 'dataset_consolidado_39_sujeitos.csv'

print("Iniciando a consolidação dos dados de EEG e Eye-Tracking...\n")

# Função auxiliar para normalização de IDs (evita divergência de formato str/int/prefixo)
def normalizar_id(series):
    s_str = series.astype(str).str.strip().str.lower()
    extraidos = s_str.str.extract(r'(\d+)')[0]
    return extraidos.apply(lambda x: str(int(x)) if pd.notna(x) and x.isdigit() else x)

# 2. Processamento dos Dados de EEG
if not os.path.exists(CAMINHO_EEG):
    CAMINHO_EEG = 'tabela_features_eeg_completa.csv'

df_eeg_raw = pd.read_csv(CAMINHO_EEG)

# Filtro estrito: apenas 'Face Feliz'
df_eeg_feliz = df_eeg_raw[df_eeg_raw['Condicao'] == 'Face Feliz'].copy()

# Média das features por participante
df_eeg_agg = df_eeg_feliz.groupby(['ID', 'Grupo']).mean(numeric_only=True).reset_index()

# Chave normalizada para o cruzamento
df_eeg_agg['ID_norm'] = normalizar_id(df_eeg_agg['ID'])

# Seleção das 5 features ouro do EEG
features_ouro_eeg = [
    'AlphaRel_P3',
    'AlphaRel_Fp2',
    'BetaRel_F7',
    'AlphaRel_F8',
    'GammaRel_F7'
]

# Adicionar prefixo nas colunas do EEG
renomear_eeg = {col: f'EEG_{col}' for col in features_ouro_eeg}
df_eeg_final = df_eeg_agg[['ID', 'ID_norm', 'Grupo'] + features_ouro_eeg].rename(columns=renomear_eeg)

print(f"✓ EEG processado ('Face Feliz'): {len(df_eeg_final)} participantes encontrados.")

# 3. Processamento dos Dados de Eye-Tracking
if not os.path.exists(CAMINHO_ET):
    CAMINHO_ET = 'dataset_features_et.csv'

df_et_raw = pd.read_csv(CAMINHO_ET)

# Chave normalizada no ET
df_et_raw['ID_norm'] = normalizar_id(df_et_raw['ID_Sujeito'])

# Identificar colunas de metadados e colunas de features de ET
cols_meta_et = ['ID_Sujeito', 'Grupo', 'Target', 'ID_norm']
feats_et = [c for c in df_et_raw.columns if c not in cols_meta_et]

print(f"✓ Eye-Tracking processado (Protocolo Geral): {len(df_et_raw)} participantes, {len(feats_et)} features.")

# 4. Alinhamento por Interseção (Inner Join via ID Normalizado)
df_consolidado = pd.merge(
    df_eeg_final,
    df_et_raw[['ID_Sujeito', 'ID_norm'] + feats_et],
    on='ID_norm',
    how='inner'
)

# Criar coluna Target unificada e limpa (1 = TEA, 0 = Controle)
df_consolidado['Target'] = df_consolidado['Grupo'].apply(lambda x: 1 if 'TEA' in str(x).upper() else 0)

# Reorganizar a ordem das colunas
cols_iniciais = ['ID_norm', 'ID_Sujeito', 'Grupo', 'Target']
cols_eeg_novas = [f'EEG_{col}' for col in features_ouro_eeg]
outras_cols = [c for c in df_consolidado.columns if c not in cols_iniciais + cols_eeg_novas]

df_consolidado = df_consolidado[cols_iniciais + cols_eeg_novas + outras_cols]

# 5. Exportação e Resumo
df_consolidado.to_csv(ARQUIVO_SAIDA, index=False)

n_total = len(df_consolidado)
n_tea = (df_consolidado['Target'] == 1).sum()
n_controle = (df_consolidado['Target'] == 0).sum()

print("\n" + "=" * 55)
print(" CONSOLIDAÇÃO CONCLUÍDA COM SUCESSO!")
print("=" * 55)
print(f"Arquivo gerado:       {ARQUIVO_SAIDA}")
print(f"Total de sujeitos:    {n_total}")
print(f"  - Grupo TEA (1):    {n_tea}")
print(f"  - Controle  (0):    {n_controle}")
print(f"Features de EEG:      {len(cols_eeg_novas)} (exclusivas 'Face Feliz')")
print(f"Features de ET:       {len(feats_et)} (protocolo completo)")
print("=" * 55)

Iniciando a consolidação dos dados de EEG e Eye-Tracking...

✓ EEG processado ('Face Feliz'): 42 participantes encontrados.
✓ Eye-Tracking processado (Protocolo Geral): 41 participantes, 252 features.

 CONSOLIDAÇÃO CONCLUÍDA COM SUCESSO!
Arquivo gerado:       dataset_consolidado_39_sujeitos.csv
Total de sujeitos:    39
  - Grupo TEA (1):    17
  - Controle  (0):    22
Features de EEG:      5 (exclusivas 'Face Feliz')
Features de ET:       252 (protocolo completo)


In [ ]:
# reservado

# Exploração

In [5]:
import os
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    f1_score
)

# 1. Carregamento dos Dados Consolidados
ARQUIVO_ENTRADA = 'dataset_consolidado_39_sujeitos.csv'

if not os.path.exists(ARQUIVO_ENTRADA):
    raise FileNotFoundError(f"Arquivo '{ARQUIVO_ENTRADA}' não encontrado! Execute a Etapa 1 primeiro.")

df = pd.read_csv(ARQUIVO_ENTRADA)

# Separação das colunas de EEG
cols_eeg = [c for c in df.columns if c.startswith('EEG_')]
cols_meta = ['ID_norm', 'ID_Sujeito', 'Grupo', 'Target']

# Separação das colunas de ET (filtrando apenas colunas numéricas para evitar colunas de texto/arquivos)
cols_et_candidatas = [c for c in df.columns if c not in cols_meta + cols_eeg]
cols_et = df[cols_et_candidatas].select_dtypes(include=[np.number]).columns.tolist()

X_eeg = df[cols_eeg].values
X_et  = df[cols_et].fillna(0).values
y     = df['Target'].values
sujeitos = df['ID_Sujeito'].values

print("=" * 65)
print(f" DADOS CARREGADOS PARA A FUSÃO TARDIA (N = {len(y)})")
print("=" * 65)
print(f"  • TEA (Classe 1):      {(y == 1).sum()} participantes")
print(f"  • Controle (Classe 0): {(y == 0).sum()} participantes")
print(f"  • Features EEG:       {len(cols_eeg)}")
print(f"  • Features ET:        {len(cols_et)}")
print("=" * 65 + "\n")

# 2. Configuração dos Modelos e do Cross-Validation Unificado
pipe_eeg = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(random_state=42))
])

model_et = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=2,
    random_state=42
)

N_FOLDS = 5
cv_strategy = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=23)

P_eeg = np.zeros(len(y))
P_et  = np.zeros(len(y))

for fold, (train_idx, test_idx) in enumerate(cv_strategy.split(X_eeg, y), 1):
    # Treino e Previsão EEG
    pipe_eeg.fit(X_eeg[train_idx], y[train_idx])
    P_eeg[test_idx] = pipe_eeg.predict_proba(X_eeg[test_idx])[:, 1]
    
    # Treino e Previsão Eye-Tracking
    model_et.fit(X_et[train_idx], y[train_idx])
    P_et[test_idx] = model_et.predict_proba(X_et[test_idx])[:, 1]

print("✓ Probabilidades out-of-fold geradas com sucesso para ambas as modalidades.\n")

# 3. Grid Search de Pesos para a Fusão Tardia
resultados = []

print("=" * 85)
print(f"{'Peso EEG':<10} | {'Peso ET':<10} | {'Acurácia':<10} | {'Acc Bal.':<10} | {'Sensib.':<10} | {'Especif.':<10} | {'AUC':<10}")
print("=" * 85)

for w_eeg in np.arange(0.0, 1.05, 0.1):
    w_eeg = round(w_eeg, 2)
    w_et  = round(1.0 - w_eeg, 2)
    
    P_fusion = (w_eeg * P_eeg) + (w_et * P_et)
    y_pred   = (P_fusion >= 0.5).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    acc     = accuracy_score(y, y_pred)
    acc_bal = balanced_accuracy_score(y, y_pred)
    sens    = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec    = tn / (tn + fp) if (tn + fp) > 0 else 0
    auc     = roc_auc_score(y, P_fusion)
    f1      = f1_score(y, y_pred, zero_division=0)
    
    resultados.append({
        'w_eeg': w_eeg,
        'w_et': w_et,
        'accuracy': acc,
        'balanced_accuracy': acc_bal,
        'sensitivity': sens,
        'specificity': spec,
        'roc_auc': auc,
        'f1_score': f1
    })
    
    print(f"{w_eeg*100:04.1f}%     | {w_et*100:04.1f}%     | {acc:.4f}   | {acc_bal:.4f}   | {sens:.4f}   | {spec:.4f}   | {auc:.4f}")

print("=" * 85 + "\n")

# 4. Identificação da Melhor Combinação e Exportação
df_res = pd.DataFrame(resultados)

melhor_config = df_res.loc[df_res['balanced_accuracy'].idxmax()]

print("=" * 65)
print(" RESULTADO DA MELHOR COMBINAÇÃO DE FUSÃO TARDIA")
print("=" * 65)
print(f"  • Peso EEG:                 {melhor_config['w_eeg']*100:.0f}%")
print(f"  • Peso Eye-Tracking:        {melhor_config['w_et']*100:.0f}%")
print(f"  • Acurácia Global:          {melhor_config['accuracy']*100:.2f}%")
print(f"  • Acurácia Balanceada:      {melhor_config['balanced_accuracy']*100:.2f}%")
print(f"  • Sensibilidade (TEA):      {melhor_config['sensitivity']*100:.2f}%")
print(f"  • Especificidade (Control): {melhor_config['specificity']*100:.2f}%")
print(f"  • ROC-AUC:                  {melhor_config['roc_auc']:.4f}")
print("=" * 65)

# Salvar relatórios
df_res.to_csv('09_fusao_tardia_metricas_pesos.csv', index=False)

P_opt = (melhor_config['w_eeg'] * P_eeg) + (melhor_config['w_et'] * P_et)
df_preds = pd.DataFrame({
    'ID_Sujeito': sujeitos,
    'Target': y,
    'P_EEG': P_eeg,
    'P_ET': P_et,
    'P_Fusao_Optima': P_opt,
    'Pred_Final': (P_opt >= 0.5).astype(int)
})
df_preds.to_csv('09_fusao_tardia_predicoes_sujeitos.csv', index=False)

print("\n✓ Relatórios salvos:")
print("   1. '09_fusao_tardia_metricas_pesos.csv'")
print("   2. '09_fusao_tardia_predicoes_sujeitos.csv'")

 DADOS CARREGADOS PARA A FUSÃO TARDIA (N = 39)
  • TEA (Classe 1):      17 participantes
  • Controle (Classe 0): 22 participantes
  • Features EEG:       5
  • Features ET:        252

✓ Probabilidades out-of-fold geradas com sucesso para ambas as modalidades.

Peso EEG   | Peso ET    | Acurácia   | Acc Bal.   | Sensib.    | Especif.   | AUC       
00.0%     | 100.0%     | 0.7949   | 0.7848   | 0.7059   | 0.8636   | 0.7286
10.0%     | 90.0%     | 0.7949   | 0.7848   | 0.7059   | 0.8636   | 0.8556
20.0%     | 80.0%     | 0.7949   | 0.7848   | 0.7059   | 0.8636   | 0.8583
30.0%     | 70.0%     | 0.7949   | 0.7848   | 0.7059   | 0.8636   | 0.8583
40.0%     | 60.0%     | 0.7949   | 0.7848   | 0.7059   | 0.8636   | 0.8636
50.0%     | 50.0%     | 0.7949   | 0.7848   | 0.7059   | 0.8636   | 0.8690
60.0%     | 40.0%     | 0.8462   | 0.8369   | 0.7647   | 0.9091   | 0.8770
70.0%     | 30.0%     | 0.7949   | 0.7914   | 0.7647   | 0.8182   | 0.8717
80.0%     | 20.0%     | 0.8205   | 0.8209   | 0

In [6]:
import os
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    classification_report
)

# 1. Carregamento dos Dados Consolidados
ARQUIVO_ENTRADA = 'dataset_consolidado_39_sujeitos.csv'

if not os.path.exists(ARQUIVO_ENTRADA):
    raise FileNotFoundError(f"Arquivo '{ARQUIVO_ENTRADA}' não encontrado! Execute a Etapa 1 primeiro.")

df = pd.read_csv(ARQUIVO_ENTRADA)

cols_eeg = [c for c in df.columns if c.startswith('EEG_')]
cols_meta = ['ID_norm', 'ID_Sujeito', 'Grupo', 'Target']
cols_et_cand = [c for c in df.columns if c not in cols_meta + cols_eeg]
cols_et = df[cols_et_cand].select_dtypes(include=[np.number]).columns.tolist()

X_eeg = df[cols_eeg].values
X_et  = df[cols_et].fillna(0).values
y     = df['Target'].values
sujeitos = df['ID_Sujeito'].values

print("=" * 65)
print(f" DADOS CARREGADOS PARA O STACKING CLASSIFIER (N = {len(y)})")
print("=" * 65)
print(f"  • TEA (Classe 1):      {(y == 1).sum()} participantes")
print(f"  • Controle (Classe 0): {(y == 0).sum()} participantes")
print(f"  • Features EEG:       {len(cols_eeg)}")
print(f"  • Features ET:        {len(cols_et)}")
print("=" * 65 + "\n")

# 2. Classe para Stacking Multimodal com Nested Out-of-Fold Validation
class MultimodalStackingClassifier:
    def __init__(self, pipe_eeg, model_et, meta_clf, cv_inner=5):
        self.pipe_eeg = pipe_eeg
        self.model_et = model_et
        self.meta_clf = meta_clf
        self.cv_inner = cv_inner
        
    def fit(self, X_eeg, X_et, y):
        # Validação cruzada interna para gerar probabilidades out-of-fold no conjunto de treino
        cv = StratifiedKFold(n_splits=self.cv_inner, shuffle=True, random_state=42)
        P_tr_eeg = np.zeros(len(y))
        P_tr_et  = np.zeros(len(y))
        
        for tr_idx, val_idx in cv.split(X_eeg, y):
            self.pipe_eeg.fit(X_eeg[tr_idx], y[tr_idx])
            P_tr_eeg[val_idx] = self.pipe_eeg.predict_proba(X_eeg[val_idx])[:, 1]
            
            self.model_et.fit(X_et[tr_idx], y[tr_idx])
            P_tr_et[val_idx] = self.model_et.predict_proba(X_et[val_idx])[:, 1]
            
        X_meta_tr = np.column_stack([P_tr_eeg, P_tr_et])
        
        # Treino do meta-modelo sobre as probabilidades out-of-fold
        self.meta_clf.fit(X_meta_tr, y)
        
        # Re-treino dos modelos base no conjunto completo de treino
        self.pipe_eeg.fit(X_eeg, y)
        self.model_et.fit(X_et, y)
        return self
        
    def predict_proba(self, X_eeg, X_et):
        P_te_eeg = self.pipe_eeg.predict_proba(X_eeg)[:, 1]
        P_te_et  = self.model_et.predict_proba(X_et)[:, 1]
        X_meta_te = np.column_stack([P_te_eeg, P_te_et])
        return self.meta_clf.predict_proba(X_meta_te)
        
    def predict(self, X_eeg, X_et):
        P_te_eeg = self.pipe_eeg.predict_proba(X_eeg)[:, 1]
        P_te_et  = self.model_et.predict_proba(X_et)[:, 1]
        X_meta_te = np.column_stack([P_te_eeg, P_te_et])
        return self.meta_clf.predict(X_meta_te)

# 3. Execução da Validação Cruzada Aninhada (Nested CV: 5-Fold Outer / 5-Fold Inner)
pipe_eeg_base = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(random_state=42))
])

model_et_base = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=2,
    random_state=42
)

meta_learner = LogisticRegression(C=1.0, random_state=42)

stacker = MultimodalStackingClassifier(
    pipe_eeg=pipe_eeg_base,
    model_et=model_et_base,
    meta_clf=meta_learner,
    cv_inner=5
)

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=23)

y_pred_stack = np.zeros(len(y))
y_prob_stack = np.zeros(len(y))

for fold, (tr_idx, te_idx) in enumerate(outer_cv.split(X_eeg, y), 1):
    stacker.fit(X_eeg[tr_idx], X_et[tr_idx], y[tr_idx])
    y_pred_stack[te_idx] = stacker.predict(X_eeg[te_idx], X_et[te_idx])
    y_prob_stack[te_idx] = stacker.predict_proba(X_eeg[te_idx], X_et[te_idx])[:, 1]

# 4. Cálculo das Métricas de Desempenho do Stacking
acc     = accuracy_score(y, y_pred_stack)
acc_bal = balanced_accuracy_score(y, y_pred_stack)
tn, fp, fn, tp = confusion_matrix(y, y_pred_stack).ravel()
sens    = tp / (tp + fn) if (tp + fn) > 0 else 0
spec    = tn / (tn + fp) if (tn + fp) > 0 else 0
auc     = roc_auc_score(y, y_prob_stack)

print("=" * 65)
print(" RESULTADOS DO STACKING CLASSIFIER (NESTED CV: 5-FOLD OUTER / 5-FOLD INNER)")
print("=" * 65)
print(f"  • Acurácia Global:          {acc*100:.2f}% ({tp+tn}/39)")
print(f"  • Acurácia Balanceada:      {acc_bal*100:.2f}%")
print(f"  • Sensibilidade (TEA):      {sens*100:.2f}% ({tp}/{tp+fn})")
print(f"  • Especificidade (Control): {spec*100:.2f}% ({tn}/{tn+fp})")
print(f"  • ROC-AUC:                  {auc:.4f}")
print("=" * 65)

 DADOS CARREGADOS PARA O STACKING CLASSIFIER (N = 39)
  • TEA (Classe 1):      17 participantes
  • Controle (Classe 0): 22 participantes
  • Features EEG:       5
  • Features ET:        252

 RESULTADOS DO STACKING CLASSIFIER (NESTED CV: 5-FOLD OUTER / 5-FOLD INNER)
  • Acurácia Global:          79.49% (31/39)
  • Acurácia Balanceada:      77.81%
  • Sensibilidade (TEA):      64.71% (11/17)
  • Especificidade (Control): 90.91% (20/22)
  • ROC-AUC:                  0.8663


In [7]:
import os
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    f1_score
)

# 1. Carregamento dos Dados Consolidados
ARQUIVO_ENTRADA = 'dataset_consolidado_39_sujeitos.csv'

if not os.path.exists(ARQUIVO_ENTRADA):
    raise FileNotFoundError(f"Arquivo '{ARQUIVO_ENTRADA}' não encontrado! Execute a Etapa 1 primeiro.")

df = pd.read_csv(ARQUIVO_ENTRADA)

# Separação das colunas por modalidade
cols_eeg = [c for c in df.columns if c.startswith('EEG_')]
cols_meta = ['ID_norm', 'ID_Sujeito', 'Grupo', 'Target']

# Seleção exclusiva de colunas numéricas do Eye-Tracking
cols_et_candidatas = [c for c in df.columns if c not in cols_meta + cols_eeg]
cols_et = df[cols_et_candidatas].select_dtypes(include=[np.number]).columns.tolist()

X_eeg = df[cols_eeg].values
X_et  = df[cols_et].fillna(0).values
y     = df['Target'].values
sujeitos = df['ID_Sujeito'].values

print("=" * 65)
print(f" DADOS CARREGADOS PARA A FUSÃO TARDIA - 10-FOLD CV (N = {len(y)})")
print("=" * 65)
print(f"  • TEA (Classe 1):      {(y == 1).sum()} participantes")
print(f"  • Controle (Classe 0): {(y == 0).sum()} participantes")
print(f"  • Features EEG:       {len(cols_eeg)}")
print(f"  • Features ET:        {len(cols_et)}")
print("=" * 65 + "\n")

# 2. Configuração dos Modelos e do Cross-Validation Unificado (10-Fold)
pipe_eeg = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(random_state=42))
])

model_et = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=2,
    random_state=42
)

# Validação Cruzada Estratificada em 10 Folds
N_FOLDS = 10
cv_strategy = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=23)

P_eeg = np.zeros(len(y))
P_et  = np.zeros(len(y))

for fold, (train_idx, test_idx) in enumerate(cv_strategy.split(X_eeg, y), 1):
    # Treino e Previsão EEG
    pipe_eeg.fit(X_eeg[train_idx], y[train_idx])
    P_eeg[test_idx] = pipe_eeg.predict_proba(X_eeg[test_idx])[:, 1]
    
    # Treino e Previsão Eye-Tracking
    model_et.fit(X_et[train_idx], y[train_idx])
    P_et[test_idx] = model_et.predict_proba(X_et[test_idx])[:, 1]

print("✓ Probabilidades out-of-fold (10-Fold CV) geradas com sucesso.\n")

# 3. Grid Search de Pesos para a Fusão Tardia
resultados = []

print("=" * 85)
print(f"{'Peso EEG':<10} | {'Peso ET':<10} | {'Acurácia':<10} | {'Acc Bal.':<10} | {'Sensib.':<10} | {'Especif.':<10} | {'AUC':<10}")
print("=" * 85)

for w_eeg in np.arange(0.0, 1.05, 0.1):
    w_eeg = round(w_eeg, 2)
    w_et  = round(1.0 - w_eeg, 2)
    
    P_fusion = (w_eeg * P_eeg) + (w_et * P_et)
    y_pred   = (P_fusion >= 0.5).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    acc     = accuracy_score(y, y_pred)
    acc_bal = balanced_accuracy_score(y, y_pred)
    sens    = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec    = tn / (tn + fp) if (tn + fp) > 0 else 0
    auc     = roc_auc_score(y, P_fusion)
    f1      = f1_score(y, y_pred, zero_division=0)
    
    resultados.append({
        'w_eeg': w_eeg,
        'w_et': w_et,
        'accuracy': acc,
        'balanced_accuracy': acc_bal,
        'sensitivity': sens,
        'specificity': spec,
        'roc_auc': auc,
        'f1_score': f1
    })
    
    print(f"{w_eeg*100:04.1f}%     | {w_et*100:04.1f}%     | {acc:.4f}   | {acc_bal:.4f}   | {sens:.4f}   | {spec:.4f}   | {auc:.4f}")

print("=" * 85 + "\n")

# 4. Identificação da Melhor Combinação e Exportação
df_res = pd.DataFrame(resultados)

melhor_config = df_res.loc[df_res['balanced_accuracy'].idxmax()]

print("=" * 65)
print(" RESULTADO DA MELHOR COMBINAÇÃO DE FUSÃO TARDIA (10-FOLD CV)")
print("=" * 65)
print(f"  • Peso EEG:                 {melhor_config['w_eeg']*100:.0f}%")
print(f"  • Peso Eye-Tracking:        {melhor_config['w_et']*100:.0f}%")
print(f"  • Acurácia Global:          {melhor_config['accuracy']*100:.2f}%")
print(f"  • Acurácia Balanceada:      {melhor_config['balanced_accuracy']*100:.2f}%")
print(f"  • Sensibilidade (TEA):      {melhor_config['sensitivity']*100:.2f}%")
print(f"  • Especificidade (Control): {melhor_config['specificity']*100:.2f}%")
print(f"  • ROC-AUC:                  {melhor_config['roc_auc']:.4f}")
print("=" * 65)

# Salvar relatórios
df_res.to_csv('09_fusao_tardia_metricas_pesos_10fold.csv', index=False)

P_opt = (melhor_config['w_eeg'] * P_eeg) + (melhor_config['w_et'] * P_et)
df_preds = pd.DataFrame({
    'ID_Sujeito': sujeitos,
    'Target': y,
    'P_EEG': P_eeg,
    'P_ET': P_et,
    'P_Fusao_Optima': P_opt,
    'Pred_Final': (P_opt >= 0.5).astype(int)
})
df_preds.to_csv('09_fusao_tardia_predicoes_sujeitos_10fold.csv', index=False)

print("\n✓ Relatórios salvos:")
print("   1. '09_fusao_tardia_metricas_pesos_10fold.csv'")
print("   2. '09_fusao_tardia_predicoes_sujeitos_10fold.csv'")

 DADOS CARREGADOS PARA A FUSÃO TARDIA - 10-FOLD CV (N = 39)
  • TEA (Classe 1):      17 participantes
  • Controle (Classe 0): 22 participantes
  • Features EEG:       5
  • Features ET:        252

✓ Probabilidades out-of-fold (10-Fold CV) geradas com sucesso.

Peso EEG   | Peso ET    | Acurácia   | Acc Bal.   | Sensib.    | Especif.   | AUC       
00.0%     | 100.0%     | 0.7692   | 0.7620   | 0.7059   | 0.8182   | 0.8489
10.0%     | 90.0%     | 0.7692   | 0.7620   | 0.7059   | 0.8182   | 0.8396
20.0%     | 80.0%     | 0.7692   | 0.7620   | 0.7059   | 0.8182   | 0.8556
30.0%     | 70.0%     | 0.7692   | 0.7620   | 0.7059   | 0.8182   | 0.8583
40.0%     | 60.0%     | 0.7692   | 0.7620   | 0.7059   | 0.8182   | 0.8663
50.0%     | 50.0%     | 0.7692   | 0.7620   | 0.7059   | 0.8182   | 0.8636
60.0%     | 40.0%     | 0.7692   | 0.7553   | 0.6471   | 0.8636   | 0.8690
70.0%     | 30.0%     | 0.7949   | 0.7848   | 0.7059   | 0.8636   | 0.8610
80.0%     | 20.0%     | 0.7949   | 0.7914   | 0